In [ ]:

import sys, os, glob, shutil, time, json
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)

os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/models/ce_bi", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()

# Веса двухбашенной лежат в выводе ядра обучения. Маркер выбран уникальный: файл
# `bi_encoder/model.safetensors` есть только там.
bi_src = os.path.dirname(glob.glob("/kaggle/input/**/bi_encoder/model.safetensors", recursive=True)[0])
for p in glob.glob(bi_src + "/*"):
    if os.path.isfile(p): shutil.copy(p, "/kaggle/working/models/ce_bi/")
log(f"веса: {bi_src} -> {sorted(os.listdir('/kaggle/working/models/ce_bi'))}")
assert json.load(open("/kaggle/working/models/ce_bi/inference_config.json")).get("kind") == "biencoder"

os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
ref = np.load(glob.glob("/kaggle/input/**/ce_bi.npy", recursive=True)[0]).astype(np.float32)
log(f"пар {len(pairs):,}, товаров {len(items):,}, эталон {ref.shape}")
assert len(ref) == len(pairs)

from src.pipeline import _item_texts, _biencoder_scores
stage = time.perf_counter()
texts = _item_texts(pairs, items, "compact")
log(f"тексты {len(texts):,} карточек за {time.perf_counter()-stage:.1f}с")

stage = time.perf_counter()
got = _biencoder_scores(pairs, texts, "models/ce_bi")
took = time.perf_counter() - stage
log(f"скоринг за {took:.1f}с ({len(pairs)/took:,.0f} пар/с, {len(texts)/took:,.0f} карточек/с)")

from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score
diff = np.abs(got - ref)
log(f"совпадение с эталоном: спирмен {spearmanr(got, ref).statistic:+.6f}, "
    f"пирсон {np.corrcoef(got, ref)[0,1]:+.6f}")
log(f"  макс. расхождение {diff.max():.2e}, среднее {diff.mean():.2e}, "
    f"доля > 1e-3: {(diff > 1e-3).mean():.4%}")
y = pairs["target"].to_numpy() if "target" in pairs else pairs["label"].to_numpy()
log(f"  PR-AUC новой ветки {average_precision_score(y, got):.6f}, эталона {average_precision_score(y, ref):.6f}")

# Пары с товаром вне items должны получать ноль, а не падать: подменяю четверть ссылок
# на несуществующие идентификаторы и проверяю, что ветка это переживает.
broken = pairs.copy()
broken["id1"] = broken["id1"].astype("int64")
mask = np.arange(len(broken)) % 4 == 0
broken.loc[mask, "id1"] = -np.arange(1, mask.sum() + 1)
sub = _biencoder_scores(broken.head(20000), texts, "models/ce_bi")
holes = (np.arange(20000) % 4 == 0)
assert np.all(sub[holes] == 0.0), "пары без карточки должны давать ноль"
assert np.allclose(sub[~holes], got[:20000][~holes], atol=1e-5), "остальные пары не должны меняться"
log(f"пропуски: {holes.sum()} пар без карточки дали ноль, остальные не изменились")
log("ПРОВЕРКА ПРОЙДЕНА" if diff.max() < 1e-2 else "РАСХОЖДЕНИЕ СЛИШКОМ БОЛЬШОЕ")
